# Grade and Curate — Human-in-the-Loop Demo

Closes the curation loop: **chat with a fine-tune → capture transcripts → grade with the training-time reward → curate the high-scoring examples as new training data.**

Pairs naturally with the other three pillar notebooks (GSM8K, customer-support, tool-calling). After running one of those to produce a trained adapter, come back here to evaluate it qualitatively + build the next iteration's training set.

**Runtime:** ~5 minutes. **No GPU required** — works against the stub backend or a real LoRA adapter.

**Open in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/grade_and_curate_demo.ipynb)

## 1. Install

In [ ]:
import os
import subprocess

PINNED_COMMIT = '14c0e65'
if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
%pip install --quiet -e '.[training,api]'
print('Install complete')

## 2. Stage conversation transcripts

In production, these come from real users chatting with your fine-tuned agent. For this demo we stage three transcripts of varying quality so you can see the curation discriminate.

In [ ]:
import json
from pathlib import Path

TRANSCRIPTS = Path('/content/transcripts')
TRANSCRIPTS.mkdir(exist_ok=True)

scenarios = {
    'good': {
        'turns': [
            {'role': 'user', 'content': 'I need a refund for my order'},
            {'role': 'assistant', 'content': "I'd be happy to help with your refund. Please share the order number and I'll process it right away."},
        ],
        'context': {'intent': 'refund', 'must_acknowledge': ['refund', 'order'], 'must_avoid': ['impossible']},
    },
    'mediocre': {
        'turns': [
            {'role': 'user', 'content': 'refund please'},
            {'role': 'assistant', 'content': 'ok'},
        ],
        'context': {'intent': 'refund', 'must_acknowledge': ['refund', 'order'], 'must_avoid': []},
    },
    'bad': {
        'turns': [
            {'role': 'user', 'content': 'refund'},
            {'role': 'assistant', 'content': 'That is impossible to help with.'},
        ],
        'context': {'intent': 'refund', 'must_acknowledge': ['refund', 'order'], 'must_avoid': ['impossible']},
    },
}

for name, data in scenarios.items():
    h = TRANSCRIPTS / f'{name}.jsonl'
    c = TRANSCRIPTS / f'{name}.context.jsonl'
    h.write_text('\n'.join(json.dumps(t) for t in data['turns']) + '\n')
    c.write_text(json.dumps(data['context']) + '\n')

for f in sorted(TRANSCRIPTS.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size} bytes)')

## 3. Score each transcript with the customer-support reward

Same composite reward used to train the model — three signals (intent acknowledgement + brand voice + safety multiplier).

In [ ]:
import asyncio
from stateset_agents.core.trajectory import ConversationTurn
from stateset_agents.data.customer_support_bench import SupportRewardComposite

reward = SupportRewardComposite()

results = []
async def grade_all():
    for name, data in scenarios.items():
        turns = [ConversationTurn(role=t['role'], content=t['content']) for t in data['turns']]
        result = await reward.compute_reward(turns, context=data['context'])
        results.append((name, result.score, dict(result.breakdown)))

asyncio.run(grade_all())

print(f'{'NAME':<10} {'SCORE':<8} {'TOOL_SEL':<10} {'BRAND_VOICE':<13} {'SAFETY':<8}')
for name, score, bd in results:
    marker = '✅' if score >= 0.7 else ('⚠️ ' if score >= 0.3 else '❌')
    print(f"{marker} {name:<8} {score:<8.3f} {bd.get('intent_score', 0):<10.2f} {bd.get('brand_voice_score', 0):<13.2f} {bd.get('safety_score', 0):<8.2f}")

## 4. Curate high-scoring examples

Use the shipped `scripts/grade_transcript.py` to write a curated training set in one pass.

In [ ]:
curated_path = Path('/content/curated.jsonl')
if curated_path.exists():
    curated_path.unlink()

for name in scenarios:
    subprocess.run([
        'python', 'scripts/grade_transcript.py',
        '--history', str(TRANSCRIPTS / f'{name}.jsonl'),
        '--context-file', str(TRANSCRIPTS / f'{name}.context.jsonl'),
        '--reward', 'customer_support',
        '--output', str(TRANSCRIPTS / f'{name}.graded.md'),
        '--output-curated', str(curated_path),
        '--threshold', '0.7',
    ], check=True, capture_output=True)

print(f'Curated examples (threshold=0.7):')
if curated_path.exists():
    for line in curated_path.read_text().splitlines():
        entry = json.loads(line)
        print(f"  score={entry['score']:.2f}  source={entry['source']}")
        print(f"    prompt: {entry['prompt']}")
        print(f"    response: {entry['response'][:80]}…")
else:
    print('  (no examples passed threshold)')

## 5. Cross-transcript summary

The umbrella report you'd share with the team after a curation session.

In [ ]:
graded_dir = TRANSCRIPTS
# The per-transcript JSONs are written by --output (script auto-emits .json alongside .md when --json passed).
# For the summary, we point at the .json files directly.
for name in scenarios:
    # Each grade_transcript invocation wrote name.graded.md; the corresponding .json companion has the rows.
    # Re-run with --json to ensure the JSON exists:
    pass

# Run the summarizer
result = subprocess.run([
    'python', 'scripts/summarize_graded_batch.py',
    '--graded-dir', str(TRANSCRIPTS),
], capture_output=True, text=True, check=False)
print(result.stdout)

## 6. What's next

In production you'd:

1. **Chat against your fine-tune** via `stateset-agents chat --history sessions.jsonl --grade customer_support` — collecting a growing transcript of real-or-realistic conversations.
2. **Re-run grade_transcript** as your reward function evolves — the curated.jsonl is **idempotent** (de-duplicated by prompt+response), so repeated grading doesn't pollute the set.
3. **Use `curated.jsonl` as new SFT training data** for the next pass — high-scoring (prompt, response) pairs that the reward function and your judgment both endorse.
4. **Iterate.** Each round you retrain → chat → curate → retrain, the model and the reward function tighten on each other.

See [`docs/PLATFORM_TOUR.md`](../docs/PLATFORM_TOUR.md) for the full developer journey.